# Save tagged-unit figures

Independent notebook that loads one opto-tagging session, selects **robustly tagged units**,
and saves **one figure per unit**. Every condition for a unit is drawn in the **same figure**
(3-column grid), with the unit's mean waveform as an inset in the top-right corner.

Selection uses `OpticalTagging.find_tagged_units`, which layers several criteria on top of a
paired t-test to avoid over-calling tagged units when trial counts are large:

- **Short-latency window** (0–6 ms) for direct activation
- **SALT test** (Kvitsiani et al. 2013) on first-spike-latency distributions
- **Effect size**: `stim_FR ≥ k × baseline_FR`
- **Reliability**: fraction of pulses evoking a spike in the window
- **Latency / jitter**: small median first-spike latency and low SD
- **Multiple-comparison correction** (Benjamini–Hochberg FDR) on both the t-test and SALT p-values

Figures are saved as **PNG by default** and rendered **in parallel** across worker processes.


In [1]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")

✅ Analysis modules loaded from: /root/capsule/src/aind_dft_ephys_analysis


In [2]:
# =============================================================================
# 2. LOAD SESSION
# =============================================================================
import glob
from optical_tagging import OpticalTagging

session_date = '863973_2026-08-25_14-18-29'
behavior_json_file = f'/root/capsule/data/ecephys_{session_date}/behavior/{session_date}.json'

# The sorted folder has an extra timestamp not contained in session_date,
# so locate it by glob (one match expected per session).
ephys_nwb_file = glob.glob(
    f'/root/capsule/data/ecephys_{session_date}_sorted_*/nwb/ecephys_{session_date}_experiment1_recording1.nwb'
)[0]

OptoTag = OpticalTagging(
    behavior_json_file=behavior_json_file,
    ephys_nwb_file=ephys_nwb_file,
)

sig_csv_path = f'/root/capsule/scratch/opto_tagging/sig_{session_date}.csv'
print(f"OptoTag ready. Significance CSV: {sig_csv_path}")

Behavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json
Number of units passing the default QC: 2204
OptoTag ready. Significance CSV: /root/capsule/scratch/opto_tagging/sig_863973_2026-08-25_14-18-29.csv


In [3]:
# =============================================================================
# 3. SELECT ROBUSTLY TAGGED UNITS (SALT + latency/jitter/reliability/effect-size)
#    Includes ALL probes / shanks / lasers. The laser name is still shown in each
#    plot's title via the condition.
#
#    NOTE: this recomputes the robust metrics on the OptoTag object (it does NOT
#    read the old significance CSV, which only had the permissive t-test).
# =============================================================================
import pandas as pd

# ---- Choose the condition to plot (all probes / all shanks are included) ----
# The condition is the 6-tuple used by find_tagged_units:
#   (target_power, location_tag, laser_name, cycle_duration, frequency, pulse_duration)
#     5.0       -> target_power     : commanded laser power (mW)
#     1.0       -> location_tag      : stimulation location / site id
#     "Laser_1" -> laser_name        : which laser was used (IGNORED for selection)
#     1.0       -> cycle_duration    : laser-on duration of each train/cycle (s)
#     10.0      -> frequency         : pulse rate within the train (Hz) -> 10 pulses/s
#     0.05      -> pulse_duration    : duration of a single pulse (s) -> 50 ms
target_cond = (5.0, 1.0, "Laser_1", 1.0, 10.0, 0.05)

# ---- Robust opto-tag criteria (tune as needed) ------------------------------
#   latency_window : short direct-activation window (0-6 ms by default)
#   effect_ratio   : require stim rate >= this x baseline rate
#   min_reliability: fraction of pulses that must evoke a spike in the window
#   max_latency    : max median first-spike latency (s)
#   max_jitter     : max SD of first-spike latency (s)
#   alpha          : applied to BH-FDR-corrected t-test AND SALT p-values
tagged_df = OptoTag.find_tagged_units(
    baseline_window=(-0.05, 0.0),
    latency_window=(0.0, 0.006),
    alpha=0.05,
    effect_ratio=2.0,
    min_abs_increase=0.0,
    min_reliability=0.2,
    max_latency=0.006,
    max_jitter=0.003,
    salt_dt=0.001,
    align_to_event="pulse",
    remove_artefacts=True,
    removal_window=0.002,
)

# Match the condition on every field EXCEPT laser_name (index 2), so units tagged
# by any laser under the same power/location/timing are all included.
IGNORE_IDX = 2  # laser_name position in the 6-tuple
match_idx = [i for i in range(6) if i != IGNORE_IDX]

def matches_ignore_laser(c):
    if not isinstance(c, (tuple, list)) or len(c) != 6:
        return False
    return all(c[i] == target_cond[i] for i in match_idx)

cond_mask = tagged_df["condition"].apply(matches_ignore_laser)

# Robustly tagged units for this condition (any probe / shank / laser).
sigrows = tagged_df[cond_mask & (tagged_df["tagged"] == True)]

# Intersect with QC-passed units on the OptoTag object.
common_units = sorted(set(OptoTag.units_passing_qc.index) & set(sigrows["unit_id"]))
print(f"Condition (laser_name ignored): {target_cond} | all probes / shanks / lasers")
print(f"Robustly tagged units to plot ({len(common_units)}): {common_units}")

# Handy diagnostic table for the selected units.
cols = ["unit_id", "probe", "shank", "reliability", "median_latency", "jitter",
        "baseline_FR_mean", "stim_FR_mean", "effect_ratio",
        "p_ttest_corr", "p_salt_corr"]
display(sigrows.sort_values("p_salt_corr")[cols].reset_index(drop=True))


Condition (laser_name ignored): (5.0, 1.0, 'Laser_1', 1.0, 10.0, 0.05) | all probes / shanks / lasers
Robustly tagged units to plot (116): [310, 311, 752, 759, 788, 792, 2025, 2036, 2059, 2109, 2115, 2116, 2149, 2185, 2200, 2203, 2208, 2209, 2212, 2215, 2225, 2230, 2233, 2238, 2239, 2240, 2241, 2245, 2250, 2255, 2257, 2259, 2263, 2264, 2272, 2277, 2284, 2288, 2296, 2300, 2303, 2314, 2315, 2317, 2322, 2324, 2326, 2331, 2334, 2344, 2366, 2558, 2586, 2587, 2588, 2609, 2613, 2629, 2630, 2634, 2685, 2689, 2692, 2706, 2735, 2803, 2811, 2845, 2946, 2947, 2960, 2983, 3000, 3014, 3020, 3023, 3031, 3032, 3033, 3258, 3269, 3326, 3329, 3335, 3385, 3393, 3400, 3403, 3453, 3467, 3502, 3603, 3620, 3629, 3656, 3663, 3665, 3673, 3696, 3700, 3705, 3718, 3736, 3757, 3809, 3814, 4045, 4062, 4077, 4106, 4159, 4227, 4270, 4273, 4280, 4320]


,unit_id,probe,shank,reliability,median_latency,jitter,baseline_FR_mean,stim_FR_mean,effect_ratio,p_ttest_corr,p_salt_corr
0,310,ProbeA-1,0,0.817,0.003204,0.000895,58.40,184.833333,3.164954,5.521474e-167,0.0
1,311,ProbeA-1,0,0.541,0.003770,0.000693,29.72,104.666667,3.521759,6.426595e-85,0.0
2,752,ProbeA-2,1,0.327,0.003604,0.000944,22.34,59.000000,2.641003,2.938864e-33,0.0
3,759,ProbeA-2,1,0.707,0.004737,0.000840,58.42,122.666667,2.099738,1.183206e-94,0.0
4,788,ProbeA-2,1,0.493,0.003461,0.000900,46.30,98.666667,2.131030,1.935135e-41,0.0
...,...,...,...,...,...,...,...,...,...,...,...
111,4227,ProbeB-4,3,0.297,0.005076,0.000799,7.38,50.666667,6.865402,1.171435e-56,0.0
112,4270,ProbeB-4,3,0.241,0.005468,0.000518,0.16,40.166667,251.041667,8.551632e-60,0.0
113,4273,ProbeB-4,3,0.318,0.004236,0.000526,7.58,54.333333,7.167986,6.022870e-62,0.0
114,4280,ProbeB-4,3,0.320,0.004250,0.000613,5.78,56.166667,9.717416,3.362314e-65,0.0


In [5]:
# =============================================================================
# 4. SAVE ONE FIGURE PER TAGGED UNIT (all conditions in one figure) — PARALLEL
# =============================================================================
# plot_raster_graph now draws EVERY condition for a unit in a single figure
# (one column per condition) and writes one file per unit. Rendering is the
# slow part (lazy NWB spike reads + matplotlib), so we fan the units out over
# several processes. Each worker opens the NWB once and reuses it.
import os
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

from opto_tagging_figure_worker import init_worker, render_unit

# Output folder for the per-unit figures (all probes / shanks for this condition).
tagged_fig_dir = f"/root/capsule/scratch/tagged_units/{session_date}/all_probes_shanks/"
os.makedirs(tagged_fig_dir, exist_ok=True)

# By default we only save PNG. Add "eps"/"pdf"/"svg" here if you also need vectors.
SAVE_FORMATS = ["png"]

# Arguments forwarded to OptoTag.plot_raster_graph for every unit.
PLOT_KWARGS = dict(
    bin_size=0.002,
    time_window=[-0.02, 0.09],
    removal_window=0.002,
    align_to_event="pulse",
    remove_artefacts=True,
    show_waveform=True,   # mean waveform in the top-right corner
)

MAX_WORKERS = 4  # tune to the number of cores available in the capsule

ctx = mp.get_context("spawn")  # spawn avoids NWB/HDF5 fork-safety issues
init_args = (behavior_json_file, ephys_nwb_file, tagged_fig_dir, PLOT_KWARGS, SAVE_FORMATS)

print(f"Rendering {len(common_units)} tagged-unit figures to {tagged_fig_dir}")
print(f"Using {MAX_WORKERS} worker processes | formats: {SAVE_FORMATS}")
with ProcessPoolExecutor(
    max_workers=MAX_WORKERS,
    mp_context=ctx,
    initializer=init_worker,
    initargs=init_args,
) as ex:
    for unit_id, ok, msg in ex.map(render_unit, common_units):
        print(f"  unit {unit_id}: {'saved' if ok else 'FAILED: ' + msg}")

print("Done saving tagged-unit figures.")


Rendering 116 tagged-unit figures to /root/capsule/scratch/tagged_units/863973_2026-08-25_14-18-29/all_probes_shanks/
Using 4 worker processes | formats: ['png']
Behavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.jsonBehavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json

Behavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json
Behavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json
Number of units passing the default QC: 2204
Number of units passing the default QC: 2204
Number of units passing the default QC: 2204Number of units passing the default QC: 2204

  unit 310: saved
  unit 311: saved
  unit 752: saved
  unit 759: saved
  unit 788: saved
  unit 792: saved
  unit 202